# Section 6 model validation: extraction, metrics, lift, and maps

This notebook exposes the complete Section 6 workflow for inspection. It reconstructs site bags and raster-derived background bags, applies the shared spatial fold plan to M0–M3, plots discrimination and presence-background ranking metrics, and maps relative suitability with spatially separate test and holdout sites.

**Interpretation:** these are presence-background models. Scores and maps represent relative suitability, not calibrated occurrence probability. Riverine and upland are never pooled here.

## 1. Setup and run controls

The defaults run both Section 6 settings and all four models. Set `REUSE_SAVED_CV=True` to review the committed workflow using the ignored prior result without refitting. Mapping defaults to riverine at a coarse stride; reduce `MAP_STRIDE` for a denser, slower map.

In [ ]:
from pathlib import Path
import json
import sys
import time
import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from rasterio.transform import xy
from rasterio.windows import Window
from sklearn.metrics import roc_curve

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'klrfome').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmarks.run_section6_comparison import (  # noqa: E402
    fold_metrics,
    model_specs,
    paired_differences,
    prepare_setting,
    run_comparison,
)
from klrfome.data.formats import Bag, BagDataset  # noqa: E402
from klrfome.io.raster_source import RasterSource  # noqa: E402
from klrfome.models.distribution import DistributionClassifier  # noqa: E402

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)
print('JAX devices:', jax.devices())

In [ ]:
CONFIG_PATH = REPO_ROOT / 'benchmarks/section6_comparison_config.json'
SETTINGS_TO_RUN = ['riverine', 'upland']
REUSE_SAVED_CV = False
SAVED_RESULTS_PATH = REPO_ROOT / 'site_data/r91_section_6_data/section6_comparison/results.json'

# Dedicated three-way diagnostic partition. Repeated CV below remains primary.
TEST_FOLD = 0
HOLDOUT_FOLD = 1
PARTITION_REPEAT = 0

# Mapping controls. Coarser stride is faster; window must be odd.
MAP_SETTING = 'riverine'
MAP_METHODS = ['M0', 'M1', 'M2', 'M3']
MAP_STRIDE = 24
MAP_WINDOW = 7
MAP_BLOCK_SIZE = 512
MAP_MIN_VALID_CELLS = 3

config = json.loads(CONFIG_PATH.read_text())
data_root = REPO_ROOT / config['data_root']
config

## 2. Extract canonical bags and reconstruct spatial backgrounds

This calls the same tracked preparation used by the command-line comparison. Site cells are deduplicated, intersected with the all-band raster mask, capped at 120 cells, and dropped below three cells. Background bags come from uniform valid raster anchors and cannot overlap site cells.

In [ ]:
prepared = {}
audit_rows = []

for setting_name in SETTINGS_TO_RUN:
    started = time.perf_counter()
    dataset, fold_plan, spatial_groups, audit = prepare_setting(
        setting_name, config['settings'][setting_name], config, data_root
    )
    prepared[setting_name] = {
        'dataset': dataset,
        'fold_plan': fold_plan,
        'spatial_groups': spatial_groups,
        'audit': audit,
    }
    audit_rows.append({
        'setting': setting_name,
        'site_bags': audit['n_site_bags'],
        'background_bags': audit['n_background_bags'],
        'mask_exclusions': len(audit['raster_alignment_exclusions']),
        'features': len(audit['feature_names']),
        'block_width': audit['block_width'],
        'folds': fold_plan.n_splits,
        'repeats': fold_plan.n_repeats,
        'preparation_seconds': time.perf_counter() - started,
    })

audit_table = pd.DataFrame(audit_rows).set_index('setting')
display(audit_table.round(3))

In [ ]:
fig, axes = plt.subplots(1, len(prepared), figsize=(7 * len(prepared), 4), squeeze=False)
for ax, (setting_name, item) in zip(axes.ravel(), prepared.items()):
    dataset = item['dataset']
    sizes = pd.DataFrame({
        'cells': [bag.n_samples for bag in dataset.collections],
        'class': ['site' if bag.label == 1 else 'background' for bag in dataset.collections],
    })
    sns.histplot(data=sizes, x='cells', hue='class', bins=24, element='step', ax=ax)
    ax.set_title(f'{setting_name.title()} retained bag sizes')
    ax.set_xlabel('Valid unique cells per bag')
plt.tight_layout()

## 3. Repeated spatial grouped validation for M0–M3

Every method receives the same bags and immutable folds. Scaling and bandwidths are fitted inside each training fold. The raw fold table includes AUC, PR AUC, continuous Boyce index, top-5% lift, timing, memory diagnostics, and convergence status.

In [ ]:
if REUSE_SAVED_CV and SAVED_RESULTS_PATH.exists():
    saved = json.loads(SAVED_RESULTS_PATH.read_text())
    cv_rows = [
        {'setting': setting_name, **row}
        for setting_name in SETTINGS_TO_RUN
        for row in saved['settings'][setting_name]['fold_results']
    ]
else:
    if REUSE_SAVED_CV:
        warnings.warn(f'Saved results not found at {SAVED_RESULTS_PATH}; refitting models.')
    cv_rows = []
    for setting_name in SETTINGS_TO_RUN:
        item = prepared[setting_name]
        rows = run_comparison(item['dataset'], item['fold_plan'], config)
        cv_rows.extend({'setting': setting_name, **row} for row in rows)

cv = pd.DataFrame(cv_rows)
assert len(cv) == len(SETTINGS_TO_RUN) * 4 * config['n_splits'] * config['n_repeats']
assert cv[['auc', 'pr_auc', 'boyce', 'top_5_percent_lift']].notna().all().all()
cv.head()

In [ ]:
summary = (
    cv.groupby(['setting', 'method'])
      .agg(
          auc_mean=('auc', 'mean'), auc_sd=('auc', 'std'),
          pr_auc_mean=('pr_auc', 'mean'),
          boyce_mean=('boyce', 'mean'),
          top5_lift_mean=('top_5_percent_lift', 'mean'),
          median_fit_seconds=('fit_seconds', 'median'),
          median_predict_seconds=('predict_seconds', 'median'),
          converged=('diagnostics', lambda values: sum(bool(v['converged']) for v in values)),
      )
      .round(4)
)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, len(SETTINGS_TO_RUN), figsize=(7 * len(SETTINGS_TO_RUN), 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, setting_name in zip(axes, SETTINGS_TO_RUN):
    subset = cv[cv['setting'] == setting_name]
    sns.boxplot(data=subset, x='method', y='auc', hue='method', legend=False, ax=ax)
    sns.stripplot(data=subset, x='method', y='auc', color='black', alpha=0.55, ax=ax)
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=1)
    ax.set_title(f'{setting_name.title()} spatial-fold AUC')
    ax.set_ylim(0.45, 1.0)
plt.tight_layout()

In [ ]:
metric_labels = {
    'pr_auc': 'PR AUC',
    'boyce': 'Continuous Boyce index',
    'top_5_percent_lift': 'Top-5% lift',
    'fit_seconds': 'Fit time (seconds)',
}
long_metrics = cv.melt(
    id_vars=['setting', 'method', 'repeat', 'fold'],
    value_vars=list(metric_labels),
    var_name='metric', value_name='value',
)
fig, axes = plt.subplots(len(SETTINGS_TO_RUN), 4, figsize=(18, 4.5 * len(SETTINGS_TO_RUN)))
axes = np.asarray(axes).reshape(len(SETTINGS_TO_RUN), 4)
for row_index, setting_name in enumerate(SETTINGS_TO_RUN):
    for col_index, metric in enumerate(metric_labels):
        ax = axes[row_index, col_index]
        subset = long_metrics[(long_metrics.setting == setting_name) & (long_metrics.metric == metric)]
        sns.boxplot(data=subset, x='method', y='value', hue='method', legend=False, ax=ax)
        sns.stripplot(data=subset, x='method', y='value', color='black', alpha=0.45, ax=ax)
        ax.set_title(f'{setting_name.title()}: {metric_labels[metric]}')
        ax.set_xlabel('')
plt.tight_layout()

### Paired fold differences

Pairing is essential because each method was evaluated on the same fold. Intervals crossing zero do not support a method difference in this Section 6 run.

In [ ]:
paired_rows = []
for setting_name in SETTINGS_TO_RUN:
    paired = paired_differences(cv[cv.setting == setting_name].drop(columns='setting').to_dict('records'))
    for method, metrics in paired.items():
        for metric, values in metrics.items():
            paired_rows.append({'setting': setting_name, 'method': method, 'metric': metric, **values})
paired_table = pd.DataFrame(paired_rows)
display(paired_table.round(4))

auc_pairs = paired_table[paired_table.metric == 'auc'].copy()
fig, ax = plt.subplots(figsize=(9, 5))
for index, row in auc_pairs.reset_index(drop=True).iterrows():
    color = {'riverine': 'tab:blue', 'upland': 'tab:green'}[row.setting]
    ax.errorbar(
        index, row.mean_difference_vs_M0,
        yerr=[[row.mean_difference_vs_M0 - row.ci_95[0]], [row.ci_95[1] - row.mean_difference_vs_M0]],
        fmt='o', color=color, capsize=4,
    )
ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(range(len(auc_pairs)), [f'{r.setting} {r.method}' for r in auc_pairs.itertuples()], rotation=30)
ax.set_ylabel('Paired AUC difference versus M0')
ax.set_title('Paired spatial-fold AUC differences with approximate 95% intervals')
plt.tight_layout()

## 4. Dedicated spatial train/test/holdout diagnostic

Repeated CV above is the primary comparison. For map inspection, this section reserves one spatial fold as a final holdout and a second fold as a test set; the remaining folds train each model. Nothing from test or holdout is used for scaling or bandwidth estimation.

In [ ]:
def three_way_partition(dataset, plan, repeat=0, test_fold=0, holdout_fold=1):
    assignments = {a.fold: a for a in plan.assignments if a.repeat == repeat}
    test_indices = set(assignments[test_fold].test_indices)
    holdout_indices = set(assignments[holdout_fold].test_indices)
    assert not test_indices & holdout_indices
    train_indices = set(range(dataset.n_locations)) - test_indices - holdout_indices
    return (
        dataset.subset(sorted(train_indices)),
        dataset.subset(sorted(test_indices)),
        dataset.subset(sorted(holdout_indices)),
    )

partition_data = {}
partition_models = {}
partition_predictions = []

for setting_name, item in prepared.items():
    train, test, holdout = three_way_partition(
        item['dataset'], item['fold_plan'], PARTITION_REPEAT, TEST_FOLD, HOLDOUT_FOLD
    )
    partition_data[setting_name] = {'train': train, 'test': test, 'holdout': holdout}
    partition_models[setting_name] = {}
    for method, spec in model_specs(config).items():
        model = DistributionClassifier(
            spec, lambda_reg=config['lambda_reg'], seed=config['seed'], round_exact_kernel=True
        ).fit(train)
        partition_models[setting_name][method] = model
        for partition_name, partition in [('test', test), ('holdout', holdout)]:
            scores = np.asarray(model.predict_bags(partition))
            labels = np.asarray(partition.labels, dtype=int)
            partition_predictions.extend(
                {
                    'setting': setting_name, 'method': method, 'partition': partition_name,
                    'bag_id': bag.id, 'label': int(label), 'score': float(score),
                }
                for bag, label, score in zip(partition.collections, labels, scores)
            )

partition_scores = pd.DataFrame(partition_predictions)
partition_metric_rows = []
for keys, group in partition_scores.groupby(['setting', 'method', 'partition']):
    setting_name, method, partition_name = keys
    partition_metric_rows.append({
        'setting': setting_name, 'method': method, 'partition': partition_name,
        **fold_metrics(group.label.to_numpy(), group.score.to_numpy()),
    })
partition_metrics = pd.DataFrame(partition_metric_rows)
display(partition_metrics.round(4))

In [ ]:
fig, axes = plt.subplots(1, len(SETTINGS_TO_RUN), figsize=(7 * len(SETTINGS_TO_RUN), 5))
axes = np.atleast_1d(axes)
for ax, setting_name in zip(axes, SETTINGS_TO_RUN):
    for method in ['M0', 'M1', 'M2', 'M3']:
        group = partition_scores[
            (partition_scores.setting == setting_name)
            & (partition_scores.method == method)
            & (partition_scores.partition == 'holdout')
        ]
        fpr, tpr, _ = roc_curve(group.label, group.score)
        auc_value = partition_metrics[
            (partition_metrics.setting == setting_name)
            & (partition_metrics.method == method)
            & (partition_metrics.partition == 'holdout')
        ].auc.iloc[0]
        ax.plot(fpr, tpr, label=f'{method} (AUC={auc_value:.3f})')
    ax.plot([0, 1], [0, 1], '--', color='grey')
    ax.set_title(f'{setting_name.title()} holdout ROC')
    ax.set_xlabel('False-positive rate')
    ax.set_ylabel('True-positive rate')
    ax.legend()
plt.tight_layout()

In [ ]:
def cumulative_lift(labels, scores, fractions=np.linspace(0.05, 1.0, 20)):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    order = np.argsort(scores)[::-1]
    baseline = labels.mean()
    return pd.DataFrame({
        'fraction': fractions,
        'lift': [labels[order[:max(1, int(np.ceil(f * len(labels))))]].mean() / baseline for f in fractions],
    })

fig, axes = plt.subplots(1, len(SETTINGS_TO_RUN), figsize=(7 * len(SETTINGS_TO_RUN), 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, setting_name in zip(axes, SETTINGS_TO_RUN):
    for method in ['M0', 'M1', 'M2', 'M3']:
        group = partition_scores[
            (partition_scores.setting == setting_name)
            & (partition_scores.method == method)
            & (partition_scores.partition == 'holdout')
        ]
        curve = cumulative_lift(group.label, group.score)
        ax.plot(curve.fraction * 100, curve.lift, marker='o', markersize=3, label=method)
    ax.axhline(1, color='grey', linestyle='--')
    ax.set_title(f'{setting_name.title()} holdout cumulative lift')
    ax.set_xlabel('Top-ranked area/bags retained (%)')
    ax.set_ylabel('Lift over holdout prevalence')
    ax.legend()
plt.tight_layout()

## 5. Lazy prediction maps with test and holdout sites

The map grid is built block-by-block from the all-band raster mask. Each map anchor is represented by the valid cells in an explicit focal window. The configurable stride controls display resolution and runtime. Models remain fitted only on the three-way training partition.

In [ ]:
def raster_source_for_setting(setting_name):
    setting = config['settings'][setting_name]
    csv_path = data_root / setting['csv']
    raster_directory = data_root / setting['raster_directory']
    columns = pd.read_csv(csv_path, nrows=0).columns
    excluded = {'Unnamed: 0', 'x', 'y', 'SITENO', 'presence'}
    feature_names = [column for column in columns if column not in excluded]
    paths = [str(raster_directory / f'{feature}.tif') for feature in feature_names]
    return RasterSource(paths, band_names=feature_names), feature_names

def _aligned_start(offset, stride):
    return ((int(offset) + stride - 1) // stride) * stride

def build_lazy_map_bags(source, feature_names, stride=24, window_size=7, block_size=512, min_cells=3):
    if window_size % 2 != 1:
        raise ValueError('window_size must be odd')
    radius = window_size // 2
    bags, centers = [], []
    for block in source.iter_windows(block_size=block_size):
        row0 = max(0, int(block.row_off) - radius)
        col0 = max(0, int(block.col_off) - radius)
        row1 = min(source.height, int(block.row_off + block.height) + radius)
        col1 = min(source.width, int(block.col_off + block.width) + radius)
        expanded = Window(col0, row0, col1 - col0, row1 - row0)
        stack, valid = source.read_window(expanded)
        rows = range(_aligned_start(block.row_off, stride), int(block.row_off + block.height), stride)
        cols = range(_aligned_start(block.col_off, stride), int(block.col_off + block.width), stride)
        for row in rows:
            for col in cols:
                local_row, local_col = row - row0, col - col0
                if not valid[local_row, local_col]:
                    continue
                r0, r1 = max(0, local_row - radius), min(valid.shape[0], local_row + radius + 1)
                c0, c1 = max(0, local_col - radius), min(valid.shape[1], local_col + radius + 1)
                window_valid = valid[r0:r1, c0:c1].ravel()
                samples = stack[:, r0:r1, c0:c1].reshape(len(feature_names), -1).T[window_valid]
                if len(samples) < min_cells:
                    continue
                bag_id = f'map-{row}-{col}'
                bags.append(Bag(samples, 0, bag_id))
                x_coord, y_coord = xy(source.transform, row, col, offset='center')
                centers.append({'bag_id': bag_id, 'row': row, 'col': col, 'x': x_coord, 'y': y_coord})
    return BagDataset(
        bags, feature_names, crs=source.crs, study_design='presence_background',
        metadata={'map_stride': stride, 'map_window': window_size},
    ), pd.DataFrame(centers)

map_source, map_feature_names = raster_source_for_setting(MAP_SETTING)
map_bags, map_centers = build_lazy_map_bags(
    map_source, map_feature_names, MAP_STRIDE, MAP_WINDOW, MAP_BLOCK_SIZE, MAP_MIN_VALID_CELLS
)
print(f'Built {map_bags.n_locations:,} map bags at stride {MAP_STRIDE}.')

In [ ]:
map_scores = map_centers.copy()
for method in MAP_METHODS:
    started = time.perf_counter()
    map_scores[method] = np.asarray(partition_models[MAP_SETTING][method].predict_bags(map_bags))
    print(f'{method}: mapped in {time.perf_counter() - started:.2f}s')
map_scores.describe()

In [ ]:
def site_centroids(dataset):
    rows = []
    for bag in dataset.collections:
        if bag.label != 1 or bag.coordinates is None:
            continue
        center = np.asarray(bag.coordinates).mean(axis=0)
        rows.append({'bag_id': bag.id, 'x': center[0], 'y': center[1]})
    return pd.DataFrame(rows)

training_sites = site_centroids(partition_data[MAP_SETTING]['train'])
test_sites = site_centroids(partition_data[MAP_SETTING]['test'])
holdout_sites = site_centroids(partition_data[MAP_SETTING]['holdout'])

fig, axes = plt.subplots(2, 2, figsize=(15, 13), constrained_layout=True)
for ax, method in zip(axes.ravel(), MAP_METHODS):
    image = ax.scatter(
        map_scores.x, map_scores.y, c=map_scores[method], s=11, marker='s',
        cmap='viridis', vmin=0, vmax=1, linewidths=0, rasterized=True,
    )
    ax.scatter(training_sites.x, training_sites.y, s=13, c='white', edgecolors='black', label='training sites')
    ax.scatter(test_sites.x, test_sites.y, s=46, marker='^', c='orange', edgecolors='black', label='test sites')
    ax.scatter(holdout_sites.x, holdout_sites.y, s=70, marker='*', c='red', edgecolors='black', label='holdout sites')
    ax.set_title(f'{MAP_SETTING.title()} {method} relative suitability')
    ax.set_aspect('equal')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.legend(loc='best', fontsize=8)
    fig.colorbar(image, ax=ax, shrink=0.8, label='Relative suitability score')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
map_scores[MAP_METHODS].plot(kind='hist', bins=35, alpha=0.45, ax=axes[0])
axes[0].set_title(f'{MAP_SETTING.title()} mapped score distributions')
axes[0].set_xlabel('Relative suitability score')
sns.heatmap(map_scores[MAP_METHODS].corr(), annot=True, vmin=0, vmax=1, cmap='mako', ax=axes[1])
axes[1].set_title('Agreement among mapped model scores')
plt.tight_layout()

## Reading the results

1. Use the repeated spatial-fold tables and paired differences for method comparison; do not rank methods from a single displayed holdout.
2. Boyce and lift are the most directly useful presence-background ranking diagnostics. PR AUC depends on the constructed background prevalence.
3. Inspect whether high suitability forms coherent environmental/spatial patterns and whether test and holdout sites fall within those patterns.
4. Map stride changes display resolution, not model fitting. A denser map is computationally expensive, especially for exact M0.
5. Do not remove M3 or promote M2 from Section 6 alone. Additional physio-sheds and the pooling decision remain necessary for general conclusions.